# EDA — Brain Tumor Classification
**Dataset:** [Kaggle — Brain Tumor Classification MRI](https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri)

Clases: `glioma_tumor` | `meningioma_tumor` | `no_tumor` | `pituitary_tumor`

In [ ]:
# Instalar dependencias
!pip install -q kaggle Pillow matplotlib seaborn scikit-learn numpy pandas tqdm

## 1. Descarga del Dataset desde Kaggle

Sube tu archivo `kaggle.json` cuando se solicite.

In [ ]:
import os
from google.colab import files

# Subir kaggle.json
print("Sube tu archivo kaggle.json:")
uploaded = files.upload()

# Configurar credenciales
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb") as f:
    f.write(uploaded["kaggle.json"])
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

# Descargar y descomprimir dataset
!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri --unzip -p ./data
print("Dataset descargado en ./data/")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

# Rutas
DATA_DIR = Path("./data")
TRAIN_DIR = DATA_DIR / "Training"
TEST_DIR  = DATA_DIR / "Testing"
CLASES    = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]

os.makedirs("./outputs/figures", exist_ok=True)

print("Clases encontradas (Training):", [d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
print("Clases encontradas (Testing):",  [d.name for d in TEST_DIR.iterdir()  if d.is_dir()])

## 2. Tamaño del Dataset

In [ ]:
def contar_imagenes(base_dir, clases):
    conteo = {}
    for clase in clases:
        imgs = list((base_dir / clase).glob("*"))
        imgs = [p for p in imgs if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        conteo[clase] = len(imgs)
    return conteo

conteo_train = contar_imagenes(TRAIN_DIR, CLASES)
conteo_test  = contar_imagenes(TEST_DIR,  CLASES)

df_conteo = pd.DataFrame({
    "Clase":    CLASES,
    "Train":    [conteo_train[c] for c in CLASES],
    "Test":     [conteo_test[c]  for c in CLASES],
})
df_conteo["Total"] = df_conteo["Train"] + df_conteo["Test"]

print(df_conteo.to_string(index=False))
print(f"\nTotal imágenes train : {df_conteo['Train'].sum()}")
print(f"Total imágenes test  : {df_conteo['Test'].sum()}")
print(f"Total general        : {df_conteo['Total'].sum()}")

## 3. Balance de Clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colores = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for ax, split, conteo in zip(axes, ["Train", "Test"], [conteo_train, conteo_test]):
    valores = [conteo[c] for c in CLASES]
    bars = ax.bar(CLASES, valores, color=colores)
    ax.set_title(f"Distribución de clases — {split}", fontsize=13)
    ax.set_xlabel("Clase")
    ax.set_ylabel("Número de imágenes")
    ax.set_xticklabels(CLASES, rotation=15, ha="right")
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                str(val), ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("./outputs/figures/balance_clases.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en outputs/figures/balance_clases.png")

## 4. Análisis de Dimensionalidad — Resolución de Imágenes

In [ ]:
def analizar_dimensiones(base_dir, clases, muestra=50):
    registros = []
    for clase in clases:
        imagenes = list((base_dir / clase).glob("*"))
        imagenes = [p for p in imagenes if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        for img_path in imagenes[:muestra]:
            with Image.open(img_path) as img:
                w, h = img.size
                modo = img.mode
                canales = len(img.getbands())
                registros.append({"clase": clase, "ancho": w, "alto": h,
                                   "modo": modo, "canales": canales})
    return pd.DataFrame(registros)

df_dims = analizar_dimensiones(TRAIN_DIR, CLASES, muestra=80)

print("=== Estadísticas de resolución (Training, muestra 80 por clase) ===\n")
print(df_dims.groupby("clase")[["ancho", "alto"]].agg(["mean", "min", "max"]).round(1))
print(f"\nModos de color encontrados: {df_dims['modo'].unique()}")
print(f"Número de canales: {df_dims['canales'].unique()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dim, label in zip(axes, ["ancho", "alto"], ["Ancho (px)", "Alto (px)"]):
    for clase, color in zip(CLASES, colores):
        subset = df_dims[df_dims["clase"] == clase][dim]
        ax.hist(subset, bins=20, alpha=0.6, label=clase, color=color)
    ax.set_title(f"Distribución de {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("./outputs/figures/histograma_resolucion.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Análisis Estadístico de Intensidades de Píxeles

In [ ]:
def estadisticas_intensidad(base_dir, clases, muestra=30):
    stats = []
    for clase in clases:
        imagenes = list((base_dir / clase).glob("*"))
        imagenes = [p for p in imagenes if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        pixeles = []
        for img_path in imagenes[:muestra]:
            with Image.open(img_path) as img:
                arr = np.array(img.convert("L")).flatten()
                pixeles.extend(arr.tolist())
        pixeles = np.array(pixeles)
        stats.append({
            "clase":  clase,
            "media":  round(pixeles.mean(), 2),
            "std":    round(pixeles.std(),  2),
            "min":    int(pixeles.min()),
            "max":    int(pixeles.max()),
            "mediana":round(np.median(pixeles), 2),
        })
    return pd.DataFrame(stats)

df_stats = estadisticas_intensidad(TRAIN_DIR, CLASES, muestra=30)
print(df_stats.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

datos_plot = []
etiquetas  = []
for clase, color in zip(CLASES, colores):
    imagenes = list((TRAIN_DIR / clase).glob("*"))
    imagenes = [p for p in imagenes if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    pixeles  = []
    for img_path in imagenes[:20]:
        with Image.open(img_path) as img:
            pixeles.extend(np.array(img.convert("L")).flatten().tolist())
    datos_plot.append(np.array(pixeles))
    etiquetas.append(clase)

bp = ax.boxplot(datos_plot, labels=etiquetas, patch_artist=True)
for patch, color in zip(bp["boxes"], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Distribución de intensidades de píxeles por clase (escala de grises)")
ax.set_ylabel("Intensidad (0–255)")
ax.set_xticklabels(etiquetas, rotation=15, ha="right")
plt.tight_layout()
plt.savefig("./outputs/figures/boxplot_intensidades.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Muestras Visuales por Clase

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 13))

for fila, clase in enumerate(CLASES):
    imagenes = list((TRAIN_DIR / clase).glob("*"))
    imagenes = [p for p in imagenes if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    muestra  = imagenes[:5]
    for col, img_path in enumerate(muestra):
        ax = axes[fila][col]
        with Image.open(img_path) as img:
            ax.imshow(img, cmap="gray" if img.mode == "L" else None)
        ax.axis("off")
        if col == 0:
            ax.set_title(clase, fontsize=10, fontweight="bold", loc="left")

plt.suptitle("Muestras por clase — Training", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("./outputs/figures/muestras_por_clase.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Imagen Promedio por Clase

In [ ]:
def imagen_promedio(base_dir, clase, muestra=100, size=(224, 224)):
    imagenes = list((base_dir / clase).glob("*"))
    imagenes = [p for p in imagenes if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    arrays = []
    for img_path in imagenes[:muestra]:
        with Image.open(img_path) as img:
            arr = np.array(img.convert("RGB").resize(size), dtype=np.float32)
            arrays.append(arr)
    return np.mean(arrays, axis=0).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, clase, color in zip(axes, CLASES, colores):
    avg = imagen_promedio(TRAIN_DIR, clase)
    ax.imshow(avg)
    ax.set_title(clase, fontsize=10, fontweight="bold")
    ax.axis("off")

plt.suptitle("Imagen promedio por clase (224×224, 100 muestras)", fontsize=13)
plt.tight_layout()
plt.savefig("./outputs/figures/imagen_promedio.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Distribución de Canales RGB por Clase

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 14))
canal_nombres = ["Rojo (R)", "Verde (G)", "Azul (B)"]
canal_colores = ["#e74c3c", "#2ecc71", "#3498db"]

for fila, clase in enumerate(CLASES):
    imagenes = list((TRAIN_DIR / clase).glob("*"))
    imagenes = [p for p in imagenes if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    acumulado = [[], [], []]
    for img_path in imagenes[:40]:
        with Image.open(img_path) as img:
            arr = np.array(img.convert("RGB"))
            for c in range(3):
                acumulado[c].extend(arr[:, :, c].flatten().tolist())
    for col in range(3):
        ax = axes[fila][col]
        ax.hist(acumulado[col], bins=50, color=canal_colores[col], alpha=0.8, density=True)
        ax.set_title(f"{clase}\n{canal_nombres[col]}", fontsize=9)
        ax.set_xlabel("Intensidad (0–255)")
        ax.set_ylabel("Densidad")

plt.suptitle("Distribución de canales RGB por clase", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("./outputs/figures/canales_rgb.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Análisis de Aspecto Ratio

In [ ]:
df_dims["aspect_ratio"] = df_dims["ancho"] / df_dims["alto"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot aspecto ratio por clase
bp = axes[0].boxplot(
    [df_dims[df_dims["clase"] == c]["aspect_ratio"].values for c in CLASES],
    labels=CLASES, patch_artist=True
)
for patch, color in zip(bp["boxes"], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].axhline(1.0, color="red", linestyle="--", linewidth=1, label="Cuadrada (1:1)")
axes[0].set_title("Aspecto ratio por clase")
axes[0].set_ylabel("Ancho / Alto")
axes[0].set_xticklabels(CLASES, rotation=15, ha="right")
axes[0].legend()

# Scatter ancho vs alto
for clase, color in zip(CLASES, colores):
    sub = df_dims[df_dims["clase"] == clase]
    axes[1].scatter(sub["ancho"], sub["alto"], alpha=0.5, label=clase, color=color, s=20)
axes[1].plot([0, 600], [0, 600], "r--", linewidth=1, label="1:1")
axes[1].set_title("Ancho vs Alto por imagen")
axes[1].set_xlabel("Ancho (px)")
axes[1].set_ylabel("Alto (px)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("./outputs/figures/aspecto_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nAspecto ratio promedio por clase:")
print(df_dims.groupby("clase")["aspect_ratio"].agg(["mean", "min", "max"]).round(3))

## 10. Detección de Imágenes Atípicas

In [ ]:
def detectar_atipicas(base_dir, clases):
    reporte = []
    for clase in clases:
        imagenes = list((base_dir / clase).glob("*"))
        imagenes = [p for p in imagenes if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        for img_path in imagenes:
            try:
                with Image.open(img_path) as img:
                    arr = np.array(img.convert("L"))
                    media = arr.mean()
                    std   = arr.std()
                    w, h  = img.size
                    flags = []
                    if media < 5:
                        flags.append("casi_negra")
                    if media > 250:
                        flags.append("casi_blanca")
                    if std < 5:
                        flags.append("sin_variacion")
                    if w < 50 or h < 50:
                        flags.append("muy_pequeña")
                    if flags:
                        reporte.append({"clase": clase, "archivo": img_path.name,
                                        "media": round(media, 1), "std": round(std, 1),
                                        "ancho": w, "alto": h, "flags": ", ".join(flags)})
            except Exception as e:
                reporte.append({"clase": clase, "archivo": img_path.name,
                                "media": None, "std": None, "ancho": None,
                                "alto": None, "flags": f"error: {e}"})
    return pd.DataFrame(reporte)

df_atipicas = detectar_atipicas(TRAIN_DIR, CLASES)
if df_atipicas.empty:
    print("✓ No se encontraron imágenes atípicas o corruptas.")
else:
    print(f"⚠ Imágenes atípicas encontradas: {len(df_atipicas)}")
    print(df_atipicas.to_string(index=False))

## 11. Pie Chart — Balance de Clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, split, conteo in zip(axes, ["Train", "Test"], [conteo_train, conteo_test]):
    valores = [conteo[c] for c in CLASES]
    wedges, texts, autotexts = ax.pie(
        valores,
        labels=CLASES,
        colors=colores,
        autopct="%1.1f%%",
        startangle=140,
        pctdistance=0.82
    )
    for at in autotexts:
        at.set_fontsize(10)
        at.set_fontweight("bold")
    ax.set_title(f"Balance de clases — {split}\n({sum(valores)} imágenes)", fontsize=12)

plt.tight_layout()
plt.savefig("./outputs/figures/pie_balance.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Resumen del EDA y Plan de Preprocesamiento

In [ ]:
total_train = sum(conteo_train.values())
total_test  = sum(conteo_test.values())

print("=" * 55)
print("         RESUMEN EDA — Brain Tumor Classification")
print("=" * 55)
print(f"Total imágenes train  : {total_train}")
print(f"Total imágenes test   : {total_test}")
print(f"Total general         : {total_train + total_test}")
print(f"Número de clases      : {len(CLASES)}")
print()

print("Balance de clases (Train):")
for clase in CLASES:
    pct = conteo_train[clase] / total_train * 100
    print(f"  {clase:<22}: {conteo_train[clase]:>4} imágenes ({pct:.1f}%)")

print()
clase_mayor = max(conteo_train, key=conteo_train.get)
clase_menor = min(conteo_train, key=conteo_train.get)
ratio = conteo_train[clase_mayor] / conteo_train[clase_menor]
print(f"Clase mayoritaria : {clase_mayor} ({conteo_train[clase_mayor]})")
print(f"Clase minoritaria : {clase_menor} ({conteo_train[clase_menor]})")
print(f"Ratio desbalance  : {ratio:.2f}x")

print()
print("Plan de preprocesamiento propuesto:")
print("  1. Redimensionar todas las imágenes a 224x224 px")
print("  2. Convertir a RGB (3 canales) si la imagen es escala de grises")
print("  3. Normalizar píxeles: media=[0.485,0.456,0.406], std=[0.229,0.224,0.225]")
print("  4. Data augmentation (train): rotación ±15°, flip horizontal,")
print("     zoom aleatorio, brillo aleatorio")
print("  5. Manejo de desbalance: class_weight en la función de pérdida")
print("  6. Split: 80% train / 20% val sobre el conjunto Training")
print("=" * 55)

## 13. Reporte de Resultados EDA (HTML limpio)

In [ ]:
import base64
import os
from pathlib import Path
from google.colab import files

def img_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

figures_dir = Path("./outputs/figures")

# Verificar figuras disponibles
figuras = {
    "balance_clases":       figures_dir / "balance_clases.png",
    "pie_balance":          figures_dir / "pie_balance.png",
    "histograma_resolucion":figures_dir / "histograma_resolucion.png",
    "boxplot_intensidades": figures_dir / "boxplot_intensidades.png",
    "canales_rgb":          figures_dir / "canales_rgb.png",
    "imagen_promedio":      figures_dir / "imagen_promedio.png",
    "aspecto_ratio":        figures_dir / "aspecto_ratio.png",
    "muestras_por_clase":   figures_dir / "muestras_por_clase.png",
}

# Estadísticas para las tablas (tomadas de las variables ya calculadas)
resumen_conteo = df_conteo.to_html(index=False, classes="tabla", border=0)
resumen_stats  = df_stats.to_html(index=False, classes="tabla", border=0)

# Construir imágenes en base64 (solo las que existen)
imgs = {}
for key, path in figuras.items():
    if path.exists():
        imgs[key] = img_to_base64(path)
    else:
        imgs[key] = None

html = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>EDA — Brain Tumor Classification</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Segoe UI', Arial, sans-serif; background: #f4f6f9;
          color: #222; padding: 40px 20px; }}
  .container {{ max-width: 960px; margin: 0 auto; }}
  h1 {{ font-size: 2rem; color: #1a237e; border-bottom: 3px solid #1a237e;
        padding-bottom: 12px; margin-bottom: 8px; }}
  .subtitle {{ color: #555; font-size: 0.95rem; margin-bottom: 40px; }}
  h2 {{ font-size: 1.3rem; color: #283593; margin: 40px 0 8px; }}
  h3 {{ font-size: 1rem; color: #444; margin: 20px 0 6px; }}
  p  {{ line-height: 1.7; color: #333; margin-bottom: 12px; font-size: 0.95rem; }}
  .card {{ background: white; border-radius: 10px; padding: 28px;
           box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin-bottom: 32px; }}
  img {{ width: 100%; border-radius: 6px; margin: 14px 0; }}
  .tabla-wrap {{ overflow-x: auto; margin: 16px 0; }}
  table.tabla {{ width: 100%; border-collapse: collapse; font-size: 0.9rem; }}
  table.tabla th {{ background: #1a237e; color: white; padding: 10px 14px;
                    text-align: left; }}
  table.tabla td {{ padding: 9px 14px; border-bottom: 1px solid #e0e0e0; }}
  table.tabla tr:nth-child(even) td {{ background: #f5f5f5; }}
  .insight {{ background: #e8eaf6; border-left: 4px solid #3949ab;
              padding: 14px 18px; border-radius: 0 8px 8px 0;
              margin: 16px 0; font-size: 0.93rem; }}
  .badge {{ display: inline-block; background: #3949ab; color: white;
            padding: 3px 10px; border-radius: 12px; font-size: 0.8rem;
            margin: 2px; }}
  footer {{ text-align: center; color: #888; font-size: 0.85rem;
            margin-top: 60px; padding-top: 20px;
            border-top: 1px solid #ddd; }}
</style>
</head>
<body>
<div class="container">

  <h1>Reporte EDA — Brain Tumor Classification</h1>
  <p class="subtitle">
    Dataset: <a href="https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri" target="_blank">
    Kaggle — Brain Tumor Classification MRI</a> &nbsp;|&nbsp;
    Clases: <span class="badge">glioma_tumor</span>
             <span class="badge">meningioma_tumor</span>
             <span class="badge">no_tumor</span>
             <span class="badge">pituitary_tumor</span>
  </p>

  <!-- SECCIÓN 1: TAMAÑO -->
  <div class="card">
    <h2>1. Tamaño del Dataset</h2>
    <p>El dataset cuenta con <strong>3,264 imágenes MRI</strong> distribuidas en cuatro clases,
    divididas en conjuntos de entrenamiento y prueba. La siguiente tabla resume la distribución:</p>
    <div class="tabla-wrap">{resumen_conteo}</div>
    <div class="insight">
      El conjunto de entrenamiento concentra el 87.9% de las imágenes (2,870),
      mientras que el conjunto de prueba representa el 12.1% restante (394).
    </div>
  </div>

  <!-- SECCIÓN 2: BALANCE -->
  <div class="card">
    <h2>2. Balance de Clases</h2>
    <p>Se analiza la distribución de imágenes por clase en ambos conjuntos para identificar
    posibles sesgos que puedan afectar el entrenamiento del modelo.</p>
    {"<img src='data:image/png;base64," + imgs['balance_clases'] + "'/>" if imgs['balance_clases'] else ""}
    {"<img src='data:image/png;base64," + imgs['pie_balance'] + "'/>" if imgs['pie_balance'] else ""}
    <div class="insight">
      <strong>Desbalance identificado:</strong> La clase <em>no_tumor</em> es la
      minoritaria en el conjunto de entrenamiento, con aproximadamente la mitad de imágenes
      respecto a las clases de tumor. Esto requiere estrategias de compensación como
      <em>class_weight</em> en la función de pérdida o técnicas de oversampling durante
      el entrenamiento.
    </div>
  </div>

  <!-- SECCIÓN 3: DIMENSIONALIDAD -->
  <div class="card">
    <h2>3. Análisis de Dimensionalidad</h2>
    <p>Se inspeccionó la resolución (ancho × alto en píxeles) de una muestra de 80 imágenes
    por clase para determinar la variabilidad dimensional del dataset.</p>
    {"<img src='data:image/png;base64," + imgs['histograma_resolucion'] + "'/>" if imgs['histograma_resolucion'] else ""}
    {"<img src='data:image/png;base64," + imgs['aspecto_ratio'] + "'/>" if imgs['aspecto_ratio'] else ""}
    <div class="insight">
      Las imágenes presentan resoluciones variables entre clases. Se requiere un
      <strong>redimensionamiento uniforme a 224×224 px</strong> como paso de
      preprocesamiento obligatorio para garantizar entradas consistentes al modelo CNN.
    </div>
  </div>

  <!-- SECCIÓN 4: INTENSIDADES -->
  <div class="card">
    <h2>4. Análisis Estadístico de Intensidades</h2>
    <p>Se analizaron las distribuciones de intensidad de píxeles (escala de grises) y
    canales RGB por clase para entender las características fotométricas del dataset.</p>
    <div class="tabla-wrap">{resumen_stats}</div>
    {"<img src='data:image/png;base64," + imgs['boxplot_intensidades'] + "'/>" if imgs['boxplot_intensidades'] else ""}
    {"<img src='data:image/png;base64," + imgs['canales_rgb'] + "'/>" if imgs['canales_rgb'] else ""}
    <div class="insight">
      Las distribuciones de intensidad difieren entre clases, lo que sugiere que el modelo
      CNN puede extraer características discriminativas a nivel fotométrico.
      Se propone <strong>normalización Z-score</strong> con media [0.485, 0.456, 0.406]
      y desviación [0.229, 0.224, 0.225] (valores ImageNet) para estabilizar el entrenamiento.
    </div>
  </div>

  <!-- SECCIÓN 5: IMAGEN PROMEDIO -->
  <div class="card">
    <h2>5. Imagen Promedio por Clase</h2>
    <p>La imagen promedio de cada clase revela los patrones visuales típicos que
    caracterizan cada tipo de tumor en MRI. Se calculó como la media píxel a píxel
    de 100 imágenes redimensionadas a 224×224 px.</p>
    {"<img src='data:image/png;base64," + imgs['imagen_promedio'] + "'/>" if imgs['imagen_promedio'] else ""}
    <div class="insight">
      Las imágenes promedio muestran diferencias estructurales entre clases: los gliomas
      presentan regiones hipointensas más difusas, mientras que los tumores pituitarios
      tienden a concentrarse en zonas centrales. Esto valida la viabilidad de la
      clasificación por CNN basada en patrones espaciales.
    </div>
  </div>

  <!-- SECCIÓN 6: MUESTRAS -->
  <div class="card">
    <h2>6. Muestras Representativas por Clase</h2>
    <p>Se presentan 5 imágenes representativas de cada clase del conjunto de entrenamiento
    para ilustrar la variabilidad intra-clase y las diferencias inter-clase.</p>
    {"<img src='data:image/png;base64," + imgs['muestras_por_clase'] + "'/>" if imgs['muestras_por_clase'] else ""}
    <div class="insight">
      Se observa alta variabilidad intra-clase en orientación, escala y contraste.
      Esto justifica el uso de <strong>data augmentation</strong> (rotación ±15°,
      flip horizontal, variación de brillo) durante el entrenamiento para mejorar
      la generalización del modelo.
    </div>
  </div>

  <!-- SECCIÓN 7: PLAN DE PREPROCESAMIENTO -->
  <div class="card">
    <h2>7. Plan de Preprocesamiento Propuesto</h2>
    <p>Con base en los hallazgos del EDA, se propone el siguiente pipeline de
    preprocesamiento previo al modelado:</p>
    <table class="tabla">
      <thead><tr><th>#</th><th>Paso</th><th>Justificación</th></tr></thead>
      <tbody>
        <tr><td>1</td><td>Redimensionar a 224×224 px</td>
            <td>Resoluciones variables requieren entrada uniforme</td></tr>
        <tr><td>2</td><td>Convertir a RGB (3 canales)</td>
            <td>Algunas imágenes están en escala de grises</td></tr>
        <tr><td>3</td><td>Normalización Z-score</td>
            <td>Estabiliza el gradiente durante el entrenamiento</td></tr>
        <tr><td>4</td><td>Data augmentation (train)</td>
            <td>Alta variabilidad intra-clase y dataset moderado</td></tr>
        <tr><td>5</td><td>class_weight en pérdida</td>
            <td>Desbalance entre no_tumor y clases de tumor</td></tr>
        <tr><td>6</td><td>Split 80/20 sobre Training</td>
            <td>Separar validación del conjunto de prueba oficial</td></tr>
      </tbody>
    </table>
  </div>

  <footer>
    Maestría en Analítica para Inteligencia de Negocios — Pontificia Universidad Javeriana<br>
    Aprendizaje Profundo — Proyecto Final 2026
  </footer>

</div>
</body>
</html>"""

# Guardar y descargar
with open("/content/reporte_eda.html", "w", encoding="utf-8") as f:
    f.write(html)

print("✓ Reporte generado: /content/reporte_eda.html")
files.download("/content/reporte_eda.html")